<center><ins><h1>Particle Size</h1></ins></center>

<h5>Definition:</h5>
<ul>
    <li>Laser diffraction analysis, also known as laser diffraction spectroscopy, is a technology that utilizes diffraction patterns of a laser beam passed through any object ranging from nanometers to millimeters in size to quickly measure geometrical dimensions of a particle. This particle size analysis process does not depend on volumetric flow rate, the amount of particles that passes through a surface over time.</li>
    <li>Shown is the mean particle size in µm.</li>
    <li>Files are imported over .xlsx in individual experiments but combined samples.</li>
</ul>

<center>
<img src="../images/particle-size_device.png" alt="Device" width="400" height="300">
<img src="../images/particle-size_diagram.png" alt="Diagram" width="519" height="300">
<img src="../images/particle-size_example-graph.png" alt="Example-Graph" width="342" height="300">
</center>

###### **Author:** Cedric Hering-Peter
###### **Address:** AG Schulz / Botantical Institut / Am Botanischen Garten 5 / 24118 Kiel

# Imports

In [1]:
# Imports
import os, sys
sys.path.append(os.path.abspath("/Users/cedric/Documents/Career/PhD-Student/Experiments/Data-Science"))
from scripts import data_helper, plot_helper
import pandas as pd

# Variables

In [2]:
META_DATA = pd.read_csv("../data/meta_data.csv")

# Show numeric output in decimal format e.g., 2.1514
pd.options.display.float_format = '{:,.2f}'.format

# Functions

In [3]:
def custom_psize_import(file, sample_col, value_cols):
    master_df = pd.read_excel(file)
    master_df.columns = data_helper.formatting_strings(master_df.columns) # All columns uppercase

    raw_df = pd.DataFrame()
    # add data value(s) into clean data frame
    for index, name in enumerate(value_cols):
        raw_df.insert(index, name, master_df[name])

    # divide and add sample information into clean data frame
    for index, name in enumerate(META_DATA["SAMPLE_INFORMATION"].dropna()):
        new_column = master_df[sample_col].str.split("_").str[index]
        raw_df.insert(index, name, new_column)
    
    return raw_df

# Raw Import

In [4]:
# Define data path and get all associated files
data_dir = "/Users/cedric/Documents/Career/PhD-Student/Experiments/Instrument-Data/Particle-Size-Analyser_Beckman-And-Coulter/CH_Master_PSA.xlsx"
#files = data_helper.create_filelist(data_dir, skip = " ")

# Import raw data frame
raw_df = custom_psize_import(data_dir, 
                             sample_col = "SAMPLE.SAMPLE_ID",
                             value_cols = ["PERCENTILE.D50", "GEOMETRIC.MEAN", "MODES.MODES_NAME"])
# Convert data types
raw_df = data_helper.convert_types(raw_df)
# Append start controls to physiology samples    
raw_df = data_helper.copy_start_control(raw_df, META_DATA)
# Sort data
raw_df = raw_df.sort_values(by = ["EXPERIMENT_NAME", "TIME", "SAMPLE_NAME", "BIO_REP", "TEC_REP"], ignore_index = True)
raw_df.head(3)

,ID,EXPERIMENT_NAME,SAMPLE_NAME,TIME,BIO_REP,TEC_REP,PERCENTILE.D50,GEOMETRIC.MEAN,MODES.MODES_NAME
0,CH230710,AT,0.1,0,1,1,718.43,632.67,trimodal
1,CH230802,AT,0.1,0,2,1,"1,175.79",981.62,bimodal
2,CH230907,AT,0.1,0,3,1,811.55,728.29,multimodal


# Custom Cleaning

In [5]:
clean_df = pd.DataFrame(raw_df)

# Drop duplicate measurements 
clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "5-Second-Run"].index, inplace = True)
clean_df.drop(clean_df[clean_df["SAMPLE_NAME"].str.contains("-Repetition")].index, inplace = True)

# Lower species count from 7 to 5
clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "C.vulgaris"].index, inplace = True)
clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "P.duplex"].index, inplace = True)

clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "+EPS"].index, inplace = True)

print(f"{raw_df.shape[0] - clean_df.shape[0]} rows and {raw_df.shape[1] - clean_df.shape[1]} cols were cleaned.")

35 rows and 0 cols were cleaned.


# File Export

In [6]:
# Export to raw data to csv-file
export_df = pd.DataFrame(clean_df)
file_name = "../exports/Raw-Data/Particle-Sizes_Raw-Data.csv"
export_df.to_csv(file_name, encoding = "utf-8", index = False)

# Plot Visualization 

In [7]:
# Filter time before plotting
plot_df = pd.DataFrame(data_helper.filter(clean_df, time = [12]))

# Iterate through all experiment, visualize and save plots
for exp in META_DATA["EXP_ABBR"].unique():
    sub_df = plot_df[plot_df["EXPERIMENT_NAME"] == exp]
    sub_meta = META_DATA[META_DATA["EXP_ABBR"] == exp][:1].reset_index()
    
    plot_helper.visualize_boxplot(sub_df, sub_meta, # current data subset
                                  show_x_label = True,
                                  y_data = "GEOMETRIC.MEAN", 
                                  y_label = "Particle Size [µm]", # Without unit serves as file name
                                  y_ticks = [0, 1100, 100],
                                  y_labelpad = 14,)

Boxplot "Species Composition" created and saved.
Boxplot "Salinity Treatment" created and saved.
Boxplot "pH Treatment" created and saved.
Boxplot "Antibiotics Treatment" created and saved.
Boxplot "Light Treatment" created and saved.
Boxplot "Temperature Treatment" created and saved.
Boxplot "Culture Composition" created and saved.
Boxplot "Reflocculation" created and saved.
Boxplot "Bioflocculation" created and saved.


# ARCHIVE